## Notebook Workflow Structure

This notebook systematically tests Epic clinical notes annotations extraction and retrieval functionality:

---
**Cells/Workflow Order:**

| Cell | Purpose |
|------|----------|
| 1-2 | Setup (imports, random seed) |
| 3 | Temp directory setup |
| 4 | Cleanup previous outputs |
| 5 | Start ES container + credentials |
| 6-7 | Populate dummy patient data |
| 8 | Index refresh verification |
| 9 | Generate and ingest Epic clinical notes |
| 10-11 | Initialize database and logger |
| 12-13 | Create pat2vec config with epic_clinical_notes_annotations mode |
| 14-15 | Run pat2vec pipeline |
| 16-18 | Extract all features from database |
| 19 | Feature preview |
| 20 | Data retrieval test |
| 21 | End-to-end verification |
| 22-23 | Merge builder functionality |
| 24 | Cleanup |
| 25 | Final verification |

---
**Test Failure Conditions:**
- Any cell raises unhandled exception
- Elasticsearch container fails to start
- No patient IDs generated after population
- Empty DataFrame from feature extraction
- epic_clinical_notes index has fewer documents than expected
- No pretty_name_count_epic_clinical_notes_* features with positive counts

In [ ]:
import os
import random
import shutil
import sys

import numpy as np

random_seed_value = 42

np.random.seed(random_seed_value)
random.seed(random_seed_value)

In [ ]:
import os
import sys

_here = os.getcwd()
for _c in (_here, os.path.join(_here, "notebooks", "test")):
    if os.path.exists(os.path.join(_c, "nb_temp_setup.py")):
        if _c not in sys.path:
            sys.path.insert(0, _c)
        break
else:
    raise FileNotFoundError(
        "nb_temp_setup.py not found in the working directory or '",
        "notebooks/test. Run this notebook from notebooks/test or the '",
        "repository root.",
    )

from nb_temp_setup import cleanup_nb_temp_dir, setup_nb_temp_dir

nb_temp_dir, repo_root = setup_nb_temp_dir()
print(f"Notebook output directory: {nb_temp_dir}")
print(f"Repository root: {repo_root}")

In [ ]:
for dir_to_remove in ["epic_clinical_notes_annotations_test_project"]:
    if os.path.exists(dir_to_remove):
        try:
            shutil.rmtree(dir_to_remove)
        except Exception as e:
            raise RuntimeError(
                f"Failed to clean up '{dir_to_remove}' directory: {e}. "
                "Critical error - cannot start with stale data."
            ) from e

print("Previous outputs cleaned.")

In [ ]:
from nb_temp_setup import get_notebook_port_offset
from pat2vec.util.docker_elastic import ElasticContainer

port_offset = get_notebook_port_offset()
es_container = ElasticContainer(port_offset=port_offset)
es_container.stop()
print("Starting Elasticsearch container...")
if not es_container.start():
    raise RuntimeError(
        "Failed to start Elasticsearch container. Check if Docker is running."
    )
host, username, password = es_container.get_credentials()
creds_filename = "test_elastic_credentials.py"
creds_content = f"""username = "{username}"
password = "{password}"
api_key = None
hosts = ["{host}"]
"""
with open(creds_filename, "w") as f:
    f.write(creds_content)
print(f"Created '{creds_filename}' pointing to test cluster at {host}")

In [ ]:
from pat2vec.util.config_pat2vec import config_class

schema_path = "test_files/elastic_schemas.json"
# Add check for file existence in current directory or nb_temp_dir
if not os.path.exists(schema_path):
    schema_path = os.path.join(nb_temp_dir, "test_files", "elastic_schemas.json")

config_populate = config_class(
    proj_name=f"epic_clinical_notes_annotations_test_project_{os.path.basename(nb_temp_dir)}",
    credentials_path=creds_filename,
    test_schema_path=schema_path,
    testing=True,
    testing_elastic=True,
    global_start_year=2020,
    global_start_month=1,
    global_start_day=1,
    global_end_year=2023,
    global_end_month=12,
    global_end_day=31,
)

In [ ]:
from pat2vec.util.get_dummy_data_cohort_searcher import populate_elastic_with_dummy_data

print("Populating test Elasticsearch cluster with dummy data...")
patient_ids = populate_elastic_with_dummy_data(config_populate, n_patients=5)

print()
print("Population complete.")
print(f"Generated {len(patient_ids)} dummy patients.")
print(f"Patient IDs: {patient_ids}")

In [ ]:
from pat2vec.pat2vec_search.cogstack_search_methods import initialize_cogstack_client

cs = initialize_cogstack_client(config_populate)

indices = ["epr_documents", "basic_observations", "observations", "order", "pims_apps"]
print("Refreshing indices...")
cs.elastic.indices.refresh(index=indices, ignore_unavailable=True)
print("Indices refreshed.")

print()
print("Index Status:")
for index in indices:
    try:
        if cs.elastic.indices.exists(index=index):
            count = cs.elastic.count(index=index)["count"]
            print(f"  - {index:<20}: {count} documents")
        else:
            raise RuntimeError(f"Index not created: {index}")
    except Exception as e:
        raise RuntimeError(f"Error checking index {index}: {e}")

In [ ]:
import pandas as pd

from pat2vec.util.get_dummy_data_cohort_searcher import (
    generate_epic_clinical_notes_data,
)
from pat2vec.util.elasticsearch_methods import ingest_data_to_elasticsearch

print("Generating and ingesting Epic clinical notes data...")
notes_dfs = []
for pid in patient_ids:
    df = generate_epic_clinical_notes_data(
        num_rows=3,
        entered_list=[pid],
        global_start_year=int(config_populate.global_start_year),
        global_start_month=int(config_populate.global_start_month),
        global_end_year=int(config_populate.global_end_year),
        global_end_month=int(config_populate.global_end_month),
    )
    notes_dfs.append(df)

df_notes = (
    pd.concat(notes_dfs, ignore_index=True) if len(notes_dfs) > 1 else notes_dfs[0]
)
df_notes = df_notes.where(pd.notnull(df_notes), None)

ingest_data_to_elasticsearch(
    df_notes,
    "epic_clinical_notes",
    es_client=cs.elastic,
)
cs.elastic.indices.refresh(index="epic_clinical_notes")

notes_count = cs.elastic.count(index="epic_clinical_notes")["count"]
expected_notes_count = len(patient_ids) * 3
if notes_count < expected_notes_count:
    raise RuntimeError(
        f"FATAL ERROR: epic_clinical_notes index has {notes_count} documents, "
        f"expected at least {expected_notes_count}."
    )

print(f"Ingested {len(df_notes)} clinical notes for {len(patient_ids)} patients")

In [ ]:
from pat2vec.util.logger_setup import setup_logger

PROJ_NAME = f"epic_clinical_notes_annotations_test_project_{os.path.basename(nb_temp_dir)}"
DB_FILENAME = "temp_epic_clinical_notes_annotations_db.sqlite"
DB_PATH = os.path.join(PROJ_NAME, "outputs", DB_FILENAME)

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

db_connection_string = f"sqlite:///{DB_PATH}"

logger = setup_logger()
print(f"Database initialized at {DB_PATH}")

In [ ]:
from pat2vec.main_pat2vec import main

config_obj = config_class(
    proj_name=PROJ_NAME,
    credentials_path=creds_filename,
    current_path_dir="",
    main_options={"epic_clinical_notes_annotations": True},
    batch_mode=True,
    verbosity=0,
    random_seed_val=random_seed_value,
    testing=True,
    testing_elastic=True,
    dummy_medcat_model=True,
    use_controls=False,
    medcat=False,
    start_time=None,
    patient_id_column_name="client_idcode",
    annot_filter_options={},
    shuffle_pat_list=False,
    storage_backend="database",
    db_connection_string=db_connection_string,
    all_patient_list=patient_ids,
)

print("pat2vec config created with epic_clinical_notes_annotations mode enabled.")

In [ ]:
try:
    pat2vec_obj = main(
        cogstack=True,
        use_filter=False,
        json_filter_path=None,
        random_seed_val=random_seed_value,
        hostname=None,
        config_obj=config_obj,
    )
except FileNotFoundError as e:
    raise RuntimeError(f"Failed to initialize pipeline: {e}") from e
except ValueError as e:
    raise RuntimeError(f"Failed to initialize pipeline: {e}") from e
except RuntimeError as e:
    raise RuntimeError(f"Failed to initialize pipeline: {e}") from e
except Exception as e:
    raise RuntimeError(f"Failed to initialize pipeline: {e}") from e

print("pat2vec object initialized.")
print(f"Patient list: {pat2vec_obj.all_patient_list}")

In [ ]:
if not pat2vec_obj.all_patient_list:
    raise RuntimeError("No patients in patient list after initialization.")

print(f"Processing first patient: {pat2vec_obj.all_patient_list[0]}")

try:
    pat2vec_obj.pat_maker(0)
except Exception as e:
    raise RuntimeError(f"Failed to process patient 0 with pat_maker: {e}") from e

print("Patient feature extraction complete.")

In [ ]:
from pat2vec.util.helper_functions import get_all_features

all_features = get_all_features(config_obj)

if all_features.empty:
    raise RuntimeError("FATAL ERROR: get_all_features returned an empty DataFrame.")

print(f"Successfully retrieved {all_features.shape[0]} rows from database.")

In [ ]:
feature_cols = [
    c
    for c in all_features.columns
    if c.startswith("pretty_name_count_epic_clinical_notes_")
]

assert len(feature_cols) > 0, (
    f"No epic_clinical_notes annotation columns found. "
    f"Available: {list(all_features.columns)}"
)

non_null_counts = all_features[feature_cols].notna().sum()
totally_empty = non_null_counts[non_null_counts == 0]

assert (
    len(totally_empty) == 0
), f"The following annotation columns are entirely null: {list(totally_empty.index)}"

print(f"Feature columns ({len(feature_cols)}): {feature_cols}")
for col in sorted(feature_cols):
    print(f"  {col}: {all_features[col].notna().sum()} non-null values")

In [ ]:
print("Feature DataFrame preview (first 5 rows):")
all_features.head()

In [ ]:
from pat2vec.util.helper_functions import get_df_from_db

annotations_data = get_df_from_db(
    config_obj,
    "annotations",
    "ann_epic_clinical_notes",
    patient_ids=[pat2vec_obj.all_patient_list[0]],
)

assert annotations_data is not None, "Annotations data should not be None"
print(f"Retrieved {len(annotations_data)} annotation records")

In [ ]:
from pat2vec.pat2vec_get_methods.get_method_epic_clinical_notes_annotations import (
    get_current_pat_epic_clinical_notes_annotations,
)

features_data = get_current_pat_epic_clinical_notes_annotations(
    current_pat_client_id_code=pat2vec_obj.all_patient_list[0],
    target_date_range=(2020, 1, 1, 2023, 12, 31),
    epic_clinical_notes_annotations=annotations_data,
    config_obj=config_obj,
)

assert features_data is not None, "Features data should not be None"

if isinstance(features_data, list):
    feature_df = features_data[0] if len(features_data) > 0 else None
else:
    feature_df = features_data

assert (
    feature_df is not None and not feature_df.empty
), f"Features DataFrame should not be empty. Got: {type(feature_df)}"

print(f"Feature extraction returned DataFrame with shape: {feature_df.shape}")


In [ ]:
all_features_full = get_all_features(config_obj)

assert not all_features_full.empty, "Feature DataFrame is empty after full pipeline."

# Verify data retrieved matches what was previously extracted
print(f"Retrieved {len(all_features_full)} rows from database")


In [ ]:
from pat2vec.util.post_processing_build_methods import build_merged_epr_mct_annot_df

merged_annots_path = build_merged_epr_mct_annot_df(
    pat2vec_obj.all_patient_list,
    config_obj,
    overwrite=True,
)

assert merged_annots_path is not None, "Merged annotations path should not be None"
assert os.path.exists(
    merged_annots_path
), f"Merged annotations file should exist at {merged_annots_path}"

import pandas as pd

merged_annots = pd.read_csv(merged_annots_path)

assert not merged_annots.empty, "Merged annotations DataFrame should not be empty."

print(f"Merge builder produced {len(merged_annots)} rows")

In [ ]:
try:
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
except Exception as e:
    raise AssertionError(f"Failed to remove database file '{DB_PATH}': {e}") from e

try:
    if os.path.exists(PROJ_NAME):
        shutil.rmtree(PROJ_NAME, ignore_errors=False)
except Exception as e:
    raise AssertionError(f"Failed to remove '{PROJ_NAME}' directory: {e}") from e

es_container.stop()
print("ES container stopped.")

cleanup_nb_temp_dir(nb_temp_dir)
print(f"Temp directory cleaned up.")

In [ ]:
assert not os.path.exists(DB_PATH), "Database file should be removed"
assert not os.path.exists(PROJ_NAME), "Project directory should be removed"

print("\n" + "=" * 60)
print("TEST SUCCESSFUL - All assertions passed!")
print("=" * 60)